# Assignment 9 — English → Gujarati Transformer NMT with pretrained embeddings
**ILLM** · **U23AI051**

* **Data:** the same ILCI English–Gujarati splits as Assignment 8 (`../Ass-8/data/splits`, misaligned
  blocks already removed), English source → Gujarati target.
* **Bins** (English source length in words), as in Assignment 8:

| bin | sentence length |
|---|---|
| bin1 | ≤ 10 |
| bin2 | (10, 20] |
| bin3 | (20, 30) |
| bin4 | ≥ 30 |

* **Model:** [OpenNMT-py](https://github.com/OpenNMT/OpenNMT-py) Transformer, trained twice with two kinds
  of pretrained input embeddings (300-d, used as initialisation and fine-tuned):
  1. **fastText** word vectors ([wiki.en / wiki.gu](https://fasttext.cc/docs/en/pretrained-vectors.html)) — word-level tokens.
  2. **BPEmb** subword vectors ([bheinzerling/bpemb](https://github.com/bheinzerling/bpemb)) — text is first
     encoded into BPEmb's own BPE subwords, then the matching subword vectors are loaded.
* **Evaluation:** [sacreBLEU](https://github.com/mjpost/sacrebleu), overall and per bin.

## 0. Settings and imports

In [ ]:
# !pip install OpenNMT-py==3.5.1 sacrebleu bpemb   (torch 2.2, Python 3.11)
import os, re, sys, json, subprocess, urllib.request
from collections import Counter

import pandas as pd
import sacrebleu
import torch
import matplotlib.pyplot as plt
from bpemb import BPEmb

SEED = 42
DIM = 300                               # fastText and BPEmb vectors are 300-d -> d_model = 300
WORD_VOCAB = 50000                      # fastText model: max word vocabulary per language
BPE_VS = 10000                          # BPEmb vocabulary size per language
TRAIN_STEPS = 10000

SPLIT = "../Ass-8/data/splits"          # same data as the previous assignment
FT_EN = os.path.expanduser("~/illm-data/wiki.en.vec")      # 6.6 GB, downloaded once
FT_GU = "../Ass-6/vectors/wiki.gu.vec"                     # downloaded in Assignment 6
ONMT = os.path.dirname(sys.executable)  # onmt_* scripts are installed next to python
BINS = ["bin1", "bin2", "bin3", "bin4"]

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Data (Assignment 8 splits)

In [ ]:
def read(path):
    return open(path, encoding="utf-8").read().splitlines()


data = {s: pd.DataFrame({"src": read(f"{SPLIT}/{s}.en"), "tgt": read(f"{SPLIT}/{s}.gu")})
        for s in ["train", "valid", "test"]}
test = data["test"]
test["bin"] = read(f"{SPLIT}/test.bin")
test["len"] = test.src.str.split().str.len()

counts = pd.DataFrame({"test sentences": test.bin.value_counts(),
                       "avg src len": test.groupby("bin").len.mean().round(1)}).loc[BINS]
print({s: len(d) for s, d in data.items()})
counts

## 2. Shared pieces: OpenNMT config, training, translation, evaluation

Both models use the same Transformer: 6 + 6 layers, d_model = 300 (= embedding size, required by
OpenNMT-py's Transformer), 4 heads, FFN 1200, Noam schedule, label smoothing 0.1, fp16, 10k steps,
beam 5. With d_model = 300 a head has 75 dimensions, which PyTorch's fused attention kernels do
not support (they need a multiple of 8), so `self_attn_type: scaled-dot` selects OpenNMT-py's plain
attention implementation. OpenNMT-py loads the vectors for every vocabulary token found in the embedding file; the
rest start at zero. All embeddings are fine-tuned during training.

**Normalisation for BLEU.** BPEmb's preprocessing lower-cases text and maps every digit to `0`
(`re.sub(r"\d", "0", text.lower())`), so the BPEmb model can only output `0` for digits. To compare the
two systems fairly, hypotheses and references of **both** systems are passed through the same
normalisation before sacreBLEU (default `13a` tokenisation; the ILCI references are already
tokenised, so `force=True`).

In [ ]:
def onmt(cmd, *args, log=None):
    out = open(log, "w") if log else subprocess.DEVNULL
    subprocess.run([os.path.join(ONMT, cmd), *args], check=True, stdout=out, stderr=subprocess.STDOUT)


def write_config(run, train, valid, vocab_size, src_emb, tgt_emb, min_freq=0):
    cfg = f"""
save_data: {run}/data
src_vocab: {run}/vocab.src
tgt_vocab: {run}/vocab.tgt
src_vocab_size: {vocab_size}
tgt_vocab_size: {vocab_size}
src_words_min_frequency: {min_freq}
tgt_words_min_frequency: {min_freq}
overwrite: true

data:
  corpus_1:
    path_src: {train[0]}
    path_tgt: {train[1]}
    transforms: [filtertoolong]
  valid:
    path_src: {valid[0]}
    path_tgt: {valid[1]}
src_seq_length: 200
tgt_seq_length: 200

src_embeddings: {src_emb}
tgt_embeddings: {tgt_emb}
embeddings_type: word2vec

save_model: {run}/model
save_checkpoint_steps: 5000
keep_checkpoint: 2
train_steps: {TRAIN_STEPS}
valid_steps: 2000
report_every: 500
seed: {SEED}
world_size: 1
gpu_ranks: [0]
num_workers: 0

encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: 6
dec_layers: 6
heads: 4
self_attn_type: scaled-dot
hidden_size: {DIM}
word_vec_size: {DIM}
transformer_ff: {4 * DIM}
dropout: [0.1]
attention_dropout: [0.1]

batch_type: tokens
batch_size: 4096
valid_batch_size: 2048
accum_count: [2]
normalization: tokens
model_dtype: fp16
optim: adam
adam_beta2: 0.998
decay_method: noam
learning_rate: 2.0
warmup_steps: 4000
max_grad_norm: 0
label_smoothing: 0.1
param_init: 0
param_init_glorot: true
"""
    with open(f"{run}/config.yaml", "w") as f:
        f.write(cfg)
    onmt("onmt_build_vocab", "-config", f"{run}/config.yaml", "-n_sample", "-1")


def train(run):
    model = f"{run}/model_step_{TRAIN_STEPS}.pt"
    if not os.path.exists(model):      # ~1 h per model; skip if the checkpoint already exists
        onmt("onmt_train", "-config", f"{run}/config.yaml", log=f"{run}/train.log")
    for line in open(f"{run}/train.log", encoding="utf-8"):
        if any(k in line for k in ["enc:", "dec:", "Validation", f"/{TRAIN_STEPS};"]):
            print(line.split("INFO] ")[-1].rstrip())
    return model


def translate(run, model, src):
    pred = f"{run}/pred.txt"
    if not os.path.exists(pred):
        onmt("onmt_translate", "-model", model, "-src", src, "-output", pred + ".tmp", "-gpu", "0",
             "-beam_size", "5", "-self_attn_type", "scaled-dot", "-batch_type", "tokens", "-batch_size", "4096", "-max_length", "250",
             log=f"{run}/translate.log")
        os.replace(pred + ".tmp", pred)  # only a finished translation counts as cached
    return read(pred)


norm = BPEmb.preprocess                  # lower-case + digits -> 0


def evaluate(hyps, name):
    refs = [norm(t) for t in test.tgt]
    hyps = [norm(h) for h in hyps]
    rows = []
    for b in BINS + ["all"]:
        idx = test.index if b == "all" else test.index[test.bin == b]
        h, r = [hyps[i] for i in idx], [refs[i] for i in idx]
        bleu = sacrebleu.corpus_bleu(h, [r], force=True)
        rows.append({"model": name, "bin": b, "sentences": len(idx), "BLEU": round(bleu.score, 2),
                     "chrF": round(sacrebleu.corpus_chrf(h, [r]).score, 2),
                     "1/2/3/4-gram precision": "/".join(f"{p:.1f}" for p in bleu.precisions),
                     "BP": round(bleu.bp, 3), "hyp/ref len": round(bleu.sys_len / bleu.ref_len, 3)})
    print(sacrebleu.corpus_bleu(hyps, [refs], force=True).format(signature=True))
    return pd.DataFrame(rows)

## 3. Model A — fastText word embeddings

### 3.1 Word vocabulary
The ILCI text is already tokenised (punctuation split off), so tokens are whitespace-separated words.
Vocabulary: words seen at least twice in training, at most 50k per language; everything else is `<unk>`.

In [ ]:
FT = "run_fasttext"
os.makedirs(FT, exist_ok=True)


def word_vocab(lang):
    c = Counter(w for line in data["train"]["src" if lang == "en" else "tgt"] for w in line.split())
    return [w for w, n in c.most_common(WORD_VOCAB) if n >= 2]


wvocab = {lang: word_vocab(lang) for lang in ["en", "gu"]}
print({lang: len(v) for lang, v in wvocab.items()})

### 3.2 Load the fastText vectors for the vocabulary
`wiki.en.vec` (2.5 M words) and `wiki.gu.vec` (75 k words) are streamed and only the vocabulary words
are kept (exact match first, then the lower-cased word), written as small word2vec files for OpenNMT-py.

In [ ]:
if not os.path.exists(FT_EN):
    os.makedirs(os.path.dirname(FT_EN), exist_ok=True)
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.en.vec", FT_EN + ".tmp")
    os.replace(FT_EN + ".tmp", FT_EN)


def filter_vectors(vec_path, vocab, out_path):
    want = set(vocab) | {w.lower() for w in vocab}
    found = {}
    with open(vec_path, encoding="utf-8", errors="ignore") as f:
        next(f)                                     # header: "<count> <dim>"
        for line in f:
            w = line.split(" ", 1)[0]
            if w in want and w not in found:
                found[w] = line.rstrip().split(" ", 1)[1]
    out = {w: found.get(w) or found.get(w.lower()) for w in vocab}
    out = {w: v for w, v in out.items() if v is not None}
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(f"{len(out)} {DIM}\n")
        f.writelines(f"{w} {v}\n" for w, v in out.items())
    return len(out)


for lang, path in [("en", FT_EN), ("gu", FT_GU)]:
    out = f"{FT}/emb.{lang}.vec"
    if not os.path.exists(out):
        filter_vectors(path, wvocab[lang], out)
    n = int(open(out, encoding="utf-8").readline().split()[0])
    print(f"{lang}: {n} / {len(wvocab[lang])} vocabulary words have a fastText vector "
          f"({100 * n / len(wvocab[lang]):.1f}%)")

### 3.3 Train, translate, evaluate

In [ ]:
write_config(FT, (f"{SPLIT}/train.en", f"{SPLIT}/train.gu"), (f"{SPLIT}/valid.en", f"{SPLIT}/valid.gu"),
             WORD_VOCAB, f"{FT}/emb.en.vec", f"{FT}/emb.gu.vec", min_freq=2)
ft_model = train(FT)

In [ ]:
ft_hyp = translate(FT, ft_model, f"{SPLIT}/test.en")
ft_res = evaluate(ft_hyp, "fastText")
ft_res

## 4. Model B — BPEmb subword embeddings

### 4.1 Encode the text into BPEmb subwords
As the BPEmb README shows, `BPEmb(lang=..., vs=..., dim=...)` downloads the language's SentencePiece BPE
model and vectors, and `.encode()` preprocesses (lower-case, digits → 0) and segments the text.

In [ ]:
BP = "run_bpemb"
os.makedirs(BP, exist_ok=True)
bpemb = {lang: BPEmb(lang=lang, vs=BPE_VS, dim=DIM) for lang in ["en", "gu"]}

for lang, col in [("en", "src"), ("gu", "tgt")]:
    for s, d in data.items():
        with open(f"{BP}/{s}.{lang}", "w", encoding="utf-8") as f:
            f.write("\n".join(" ".join(p) for p in bpemb[lang].encode(list(d[col]))) + "\n")

ex = data["train"].iloc[1]
print(ex.src, "->", " ".join(bpemb["en"].encode(ex.src)))
print(ex.tgt, "->", " ".join(bpemb["gu"].encode(ex.tgt)))
for lang, col in [("en", "src"), ("gu", "tgt")]:
    words = data["train"][col].str.split().str.len().sum()
    pieces = sum(len(l.split()) for l in read(f"{BP}/train.{lang}"))
    print(f"{lang}: {pieces / words:.2f} subwords per word")

### 4.2 Write the BPEmb vectors (all 10k subwords) as word2vec files

In [ ]:
for lang in ["en", "gu"]:
    kv = bpemb[lang].emb
    with open(f"{BP}/emb.{lang}.vec", "w", encoding="utf-8") as f:
        f.write(f"{len(kv.index_to_key)} {DIM}\n")
        for w, v in zip(kv.index_to_key, kv.vectors):
            f.write(w + " " + " ".join(f"{x:.5f}" for x in v) + "\n")
    print(lang, len(kv.index_to_key), "vectors")

### 4.3 Train, translate, evaluate

In [ ]:
write_config(BP, (f"{BP}/train.en", f"{BP}/train.gu"), (f"{BP}/valid.en", f"{BP}/valid.gu"),
             BPE_VS + 100, f"{BP}/emb.en.vec", f"{BP}/emb.gu.vec")
bp_model = train(BP)

In [ ]:
bp_hyp = [bpemb["gu"].decode(l.split()) for l in translate(BP, bp_model, f"{BP}/test.en")]
bp_res = evaluate(bp_hyp, "BPEmb")
bp_res

## 5. Bin-wise comparison

In [ ]:
res = pd.concat([ft_res, bp_res])
json.dump(res.to_dict("records"), open("ass9_results.json", "w"), indent=2, ensure_ascii=False)

# <unk> in the fastText output and source-side OOV words, per bin
test["src_oov"] = [sum(w not in set(wvocab["en"]) for w in s.split()) / len(s.split()) for s in test.src]
test["unk"] = [h.split().count("<unk>") / max(1, len(h.split())) for h in ft_hyp]
oov = (test.groupby("bin")[["src_oov", "unk"]].mean() * 100).round(2)
oov.columns = ["fastText src OOV %", "fastText <unk> in output %"]

table = res.pivot(index="bin", columns="model", values="BLEU").loc[BINS + ["all"], ["fastText", "BPEmb"]]
table["Δ (BPEmb − fastText)"] = (table.BPEmb - table.fastText).round(2)
table.join(oov)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
w = 0.38
for i, (m, c) in enumerate([("fastText", "#4C72B0"), ("BPEmb", "#DD8452")]):
    v = table.loc[BINS, m]
    ax.bar([x + (i - 0.5) * w for x in range(4)], v, w, label=f"{m} (all: {table.loc['all', m]})", color=c)
    for x, y in enumerate(v):
        ax.text(x + (i - 0.5) * w, y + 0.2, f"{y}", ha="center", fontsize=8)
ax.set_xticks(range(4), ["≤10", "11–20", "21–29", "≥30"])
ax.set_xlabel("English sentence length (words)")
ax.set_ylabel("BLEU")
ax.set_title("Bin-wise BLEU, English → Gujarati")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Sample translations from each bin

In [ ]:
pd.set_option("display.max_colwidth", 200)
test["fastText"], test["BPEmb"] = ft_hyp, bp_hyp
test[["bin", "src", "tgt", "fastText", "BPEmb"]].groupby("bin").sample(2, random_state=SEED)

## 7. Bin-wise result analysis

ANALYSIS_PLACEHOLDER